1. Setup and imports

In [ ]:
import os
from typing import List, Tuple

import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

2. Model : ResNet50 embedding + Siamese heads

In [ ]:
class EmbeddingNet(nn.Module):
    def __init__(self, embed_dim=256, pretrained=True):
        super().__init__()
        # ResNet50 backbone
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        in_features = self.backbone.fc.in_features
        # Remove classification head
        self.backbone.fc = nn.Identity()
        # Projection head to compact embedding
        self.proj = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, embed_dim),
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.proj(x)
        # L2 normalize embeddings for cosine-friendly metrics
        return F.normalize(x, p=2, dim=1)


class SiameseNet(nn.Module):
    def __init__(self, embed_dim=256, pretrained=True):
        super().__init__()
        self.encoder = EmbeddingNet(embed_dim=embed_dim, pretrained=pretrained)

    def forward_once(self, x):
        return self.encoder(x)

    def forward(self, x1, x2):
        z1 = self.forward_once(x1)
        z2 = self.forward_once(x2)
        return z1, z2


3. Loss Functions and distance utilities

In [ ]:
class ContrastiveLoss(nn.Module):
    """
    y = 1 for similar, y = 0 for dissimilar
    """
    def __init__(self, margin=1.0, distance="euclidean"):
        super().__init__()
        self.margin = margin
        self.distance = distance

    def pair_distance(self, z1, z2):
        if self.distance == "cosine":
            # Convert similarity to distance in [0, 2]
            cos_sim = F.cosine_similarity(z1, z2)
            return 1.0 - cos_sim
        else:  # euclidean
            return torch.sqrt(torch.sum((z1 - z2) ** 2, dim=1) + 1e-9)

    def forward(self, z1, z2, y):
        d = self.pair_distance(z1, z2)
        pos = y * (d ** 2)
        neg = (1 - y) * (F.relu(self.margin - d) ** 2)
        return torch.mean(pos + neg)


class TripletLoss(nn.Module):
    """
    Triplet: anchor, positive, negative
    """
    def __init__(self, margin=0.2, distance="euclidean"):
        super().__init__()
        self.margin = margin
        self.distance = distance

    def dist(self, a, b):
        if self.distance == "cosine":
            return 1.0 - F.cosine_similarity(a, b)
        else:
            return torch.sqrt(torch.sum((a - b) ** 2, dim=1) + 1e-9)

    def forward(self, za, zp, zn):
        d_ap = self.dist(za, zp)
        d_an = self.dist(za, zn)
        loss = F.relu(d_ap - d_an + self.margin)
        return torch.mean(loss)


def pairwise_distance(z1, z2, mode="cosine"):
    if mode == "cosine":
        # Return similarity in [-1, 1]; convert to distance in [0, 2]
        sim = F.cosine_similarity(z1, z2)
        return (1.0 - sim)  # lower is more similar
    else:
        return torch.sqrt(torch.sum((z1 - z2) ** 2, dim=1) + 1e-9)


4. Datasets : TAMPAR pairs and Amazon multi-views

In [ ]:
class TAMPARPairsDataset(Dataset):
    """
    For contrastive training: returns (img1, img2, label) where label=1 similar, 0 dissimilar
    You must implement your own indexing from TAMPAR annotations.
    """
    def __init__(self, pairs: List[Tuple[str, str, int]], transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p1, p2, y = self.pairs[idx]
        img1 = Image.open(p1).convert("RGB")
        img2 = Image.open(p2).convert("RGB")
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        return img1, img2, torch.tensor(y, dtype=torch.float32)


class TAMPARTripletDataset(Dataset):
    """
    For triplet training: returns (anchor, positive, negative)
    """
    def __init__(self, triplets: List[Tuple[str, str, str]], transform=None):
        self.triplets = triplets
        self.transform = transform

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        a, p, n = self.triplets[idx]
        ia = Image.open(a).convert("RGB")
        ip = Image.open(p).convert("RGB")
        ineg = Image.open(n).convert("RGB")
        if self.transform:
            ia = self.transform(ia)
            ip = self.transform(ip)
            ineg = self.transform(ineg)
        return ia, ip, ineg


class AmazonMultiViewDataset(Dataset):
    """
    For inference/eval: returns lists of images for sender and receiver parcels
    sender_views: [path1, path2, ...]
    receiver_views: [path1, path2, ...]
    label optional (1 same parcel, 0 different), for validation
    """
    def __init__(self, pairs_multi: List[Tuple[List[str], List[str], int]], transform=None):
        self.pairs_multi = pairs_multi
        self.transform = transform

    def __len__(self):
        return len(self.pairs_multi)

    def __getitem__(self, idx):
        sender_paths, receiver_paths, label = self.pairs_multi[idx]
        sender_imgs = [self.transform(Image.open(p).convert("RGB")) for p in sender_paths]
        receiver_imgs = [self.transform(Image.open(p).convert("RGB")) for p in receiver_paths]
        sender_tensor = torch.stack(sender_imgs)  # [S, C, H, W]
        receiver_tensor = torch.stack(receiver_imgs)  # [R, C, H, W]
        return sender_tensor, receiver_tensor, torch.tensor(label, dtype=torch.float32)


# New: CSV-backed pairs dataset with optional bbox cropping using columns from pairs_*_ssl.csv
class PairBBoxDataset(Dataset):
    """
    Dataset that reads (img_a, img_b, label) from a DataFrame with columns:
      - a_image_path, b_image_path
      - a_xmin, a_ymin, a_xmax, a_ymax (optional)
      - b_xmin, b_ymin, b_xmax, b_ymax (optional)
      - label_same (1 for similar, 0 for dissimilar)
    """
    def __init__(self, df: 'pd.DataFrame', root_dir: str = "", crop: bool = True, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.crop = crop
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _join(self, p: str) -> str:
        return p if os.path.isabs(p) else os.path.join(self.root_dir, p)

    def _safe_crop(self, img: Image.Image, xmin, ymin, xmax, ymax) -> Image.Image:
        W, H = img.size
        try:
            xmin = max(0, int(xmin))
            ymin = max(0, int(ymin))
            xmax = min(W, int(xmax))
            ymax = min(H, int(ymax))
        except Exception:
            return img
        if xmax <= xmin or ymax <= ymin:
            return img
        return img.crop((xmin, ymin, xmax, ymax))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        p1 = self._join(row['a_image_path'])
        p2 = self._join(row['b_image_path'])
        img1 = Image.open(p1).convert("RGB")
        img2 = Image.open(p2).convert("RGB")

        if self.crop:
            for prefix, img in (('a', img1), ('b', img2)):
                xmin_key, ymin_key, xmax_key, ymax_key = f'{prefix}_xmin', f'{prefix}_ymin', f'{prefix}_xmax', f'{prefix}_ymax'
                if xmin_key in row and ymin_key in row and xmax_key in row and ymax_key in row:
                    if prefix == 'a':
                        img1 = self._safe_crop(img1, row[xmin_key], row[ymin_key], row[xmax_key], row[ymax_key])
                    else:
                        img2 = self._safe_crop(img2, row[xmin_key], row[ymin_key], row[xmax_key], row[ymax_key])

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        y = float(row['label_same']) if 'label_same' in row else 1.0
        return img1, img2, torch.tensor(y, dtype=torch.float32)


# Utilities to load and split the provided CSVs
def load_pair_dfs(csv_paths: List[str]) -> 'pd.DataFrame':
    dfs = []
    for p in csv_paths:
        df = pd.read_csv(p)
        dfs.append(df)
    df_all = pd.concat(dfs, ignore_index=True)
    if 'split' in df_all.columns:
        df_all['split'] = df_all['split'].astype(str).str.lower()
    else:
        df_all['split'] = 'train'
    return df_all


def get_split(df: 'pd.DataFrame', split_names) -> 'pd.DataFrame':
    if isinstance(split_names, str):
        split_names = [split_names]
    names = [s.lower() for s in split_names]
    return df[df['split'].isin(names)].copy()

Transforms : augmentation + normalization

In [ ]:
im_size = 256
train_tf = transforms.Compose([
    transforms.Resize((im_size, im_size)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(im_size, scale=(0.9, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((im_size, im_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

5. Training loops (either contrastive or triplet)

In [ ]:
def train_contrastive(model, loader, optimizer, device, margin=1.0, distance="cosine"):
    model.train()
    criterion = ContrastiveLoss(margin=margin, distance=distance)
    total_loss = 0.0
    for img1, img2, y in loader:
        img1, img2, y = img1.to(device), img2.to(device), y.to(device)
        z1, z2 = model(img1, img2)
        loss = criterion(z1, z2, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * img1.size(0)
    return total_loss / len(loader.dataset)


def train_triplet(model, loader, optimizer, device, margin=0.2, distance="cosine"):
    model.train()
    criterion = TripletLoss(margin=margin, distance=distance)
    total_loss = 0.0
    for ia, ip, ineg in loader:
        ia, ip, ineg = ia.to(device), ip.to(device), ineg.to(device)
        za = model.forward_once(ia)
        zp = model.forward_once(ip)
        zn = model.forward_once(ineg)
        loss = criterion(za, zp, zn)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * ia.size(0)
    return total_loss / len(loader.dataset)


Validation for contrastive (AUC/thresholding; we compute mean distance for positive/negative as a quick check)

In [ ]:
@torch.no_grad()
def validate_contrastive(model, loader, device, distance="cosine"):
    model.eval()
    pos_dists, neg_dists = [], []
    for img1, img2, y in loader:
        img1, img2 = img1.to(device), img2.to(device)
        z1, z2 = model(img1, img2)
        d = pairwise_distance(z1, z2, mode=distance).cpu()
        for di, yi in zip(d, y):
            (pos_dists if yi.item()==1 else neg_dists).append(di.item())
    pos_mean = sum(pos_dists)/max(1, len(pos_dists))
    neg_mean = sum(neg_dists)/max(1, len(neg_dists))
    return {"pos_mean": pos_mean, "neg_mean": neg_mean}


6. Inference: multi-view aggregation and decision
We compare all view pairs between sender and receiver, aggregate distances, and apply a threshold.

In [ ]:
@torch.no_grad()
def parcel_similarity_score(model, sender_imgs, receiver_imgs, device, distance="cosine", agg="mean"):
    """
    sender_imgs: [S, C, H, W] tensor
    receiver_imgs: [R, C, H, W] tensor
    Returns an aggregated distance (lower means more similar)
    """
    model.eval()
    sender_imgs = sender_imgs.to(device)
    receiver_imgs = receiver_imgs.to(device)

    # Compute embeddings for all views
    z_s = model.encoder(sender_imgs)     # [S, D]
    z_r = model.encoder(receiver_imgs)   # [R, D]

    # All pair distances
    dists = []
    for i in range(z_s.size(0)):
        for j in range(z_r.size(0)):
            d = pairwise_distance(z_s[i:i+1], z_r[j:j+1], mode=distance).item()
            dists.append(d)

    if agg == "mean":
        score = sum(dists) / len(dists)
    elif agg == "max":
        score = max(dists)  # more conservative (if any view differs strongly)
    elif agg == "min":
        score = min(dists)  # best-case matching
    else:
        raise ValueError("agg must be one of ['mean','max','min']")

    return score, dists

Decision with a threshold:

In [ ]:
def decide_same_parcel(agg_distance, threshold):
    """
    For cosine-based distance in [0,2], typical thresholds often lie ~0.4-0.8,
    but you must calibrate on validation data.
    """
    return 1 if agg_distance <= threshold else 0


7. Putting it all together

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    embed_dim = 256
    distance = "cosine"  # or "euclidean"
    margin = 1.0
    lr = 1e-4
    batch_size = 16
    epochs = 10

    # Load annotated pairs from Kaggle + TAMPAR CSVs
    csvs = [
        "data/TAMPAR/pairs_tampar_ssl.csv",
        "data/kaggle/pairs_kaggle_ssl.csv",
    ]
    df_all = load_pair_dfs(csvs)

    # Build splits from CSV "split" column
    train_df = get_split(df_all, "train")
    val_df = get_split(df_all, ["validation", "valid", "val"])
    if len(val_df) == 0:
        # fallback: if only TAMPAR validation exists or different naming
        val_df = get_split(df_all, "validation")

    print(f"Pairs: total={len(df_all)} | train={len(train_df)} | val={len(val_df)}")

    # Datasets with bbox cropping (if bbox columns present)
    train_ds = PairBBoxDataset(train_df, transform=train_tf, crop=True)
    val_ds   = PairBBoxDataset(val_df,   transform=eval_tf,  crop=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    model = SiameseNet(embed_dim=embed_dim, pretrained=True).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # Train with contrastive
    for ep in range(epochs):
        train_loss = train_contrastive(model, train_loader, optimizer, device, margin=margin, distance=distance)
        val_stats = validate_contrastive(model, val_loader, device, distance=distance)
        print(f"Epoch {ep+1}/{epochs} | train_loss={train_loss:.4f} | val pos_mean={val_stats['pos_mean']:.3f} | val neg_mean={val_stats['neg_mean']:.3f}")

    # TODO: Calibrate threshold on validation to select operating point (e.g., scan thresholds)
    # Example (not executed here):
    # thresholds = [x/100 for x in range(20, 100)]
    # pick threshold maximizing validation F1/AUC

if __name__ == "__main__":
    main()